# Group 4 BOLT reproduction on BANKING77

Reem Alabdulla (26000605)  
Shaza Mahmod Badr (26001544)  
Amal Alhashmi (26000656)  
Roudha Alblooshi (26000603)  

The main experiment evaluates DeepUNK on five class partitions at two known-class ratios. A paired five-epoch KNNCon experiment measures the effect of enforcing the released supervision mask.

Select a T4 GPU runtime and run the setup and GPU checks before training. Completed experiment evidence is included in the repository.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from zipfile import ZipFile
from google.colab import files

root = Path("/content/BOLT_Final")
root.mkdir(exist_ok=True)
results = root / "results"
results.mkdir(exist_ok=True)
sources = {'code/analyze_results.py': '"""Verify saved predictions and regenerate every numeric table in the report."""\nimport argparse\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom statistics import mean, stdev\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef read(path, delimiter=","):\n    """Read CSV files with text labels preserved."""\n    with path.open(encoding="utf-8", newline="") as handle:\n        return list(csv.DictReader(handle, delimiter=delimiter))\n\n\ndef write(path, rows):\n    """Write stable, human-readable tables."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef score_predictions(rows, unknown):\n    """Calculate percentage metrics from test predictions with an explicit unknown label."""\n    gold = Counter(row["gold"] for row in rows)\n    predicted = Counter(row["predicted"] for row in rows)\n    correct = Counter(row["gold"] for row in rows if row["gold"] == row["predicted"])\n    labels = sorted(set(gold) | set(predicted))\n    fs = {label: 200 * correct[label] / (gold[label] + predicted[label]) for label in labels}\n    categories = Counter()\n    for row in rows:\n        if row["gold"] == row["predicted"]:\n            category = "correct"\n        elif row["gold"] == unknown:\n            category = "unknown accepted"\n        elif row["predicted"] == unknown:\n            category = "known rejected"\n        else:\n            category = "wrong known class"\n        categories[category] += 1\n    return {"K-F1": mean(value for label, value in fs.items() if label != unknown),\n            "N-F1": fs[unknown], "F1": mean(fs.values()),\n            "ACC": 100 * sum(correct.values()) / len(rows)}, categories\n\n\ndef all_tables(output):\n    """Check each saved score and calculate cross-fold, paired, runtime, and error tables."""\n    published = {(r["method"], r["known_ratio"]): r for r in read(ROOT / "code/published.csv")}\n    audit, runtime, errors, paired, comparisons, folds, examples = [], [], [], [], [], [], []\n    for record in sorted(output.rglob("run.json")):\n        if "work" in record.relative_to(output).parts:\n            continue\n        directory = record.parent\n        config = json.loads(record.read_text())\n        command = config["command"]\n        epochs = int(command[command.index("--num_train_epochs") + 1])\n        name, ratio, fold = directory.name.split("_")\n        is_deepunk = name == "deepunk"\n        duration = float(config["elapsed_seconds"])\n        runtime.append(dict(trial=directory.relative_to(output).as_posix(), seconds=duration,\n                            epoch_cap=epochs))\n        for row in read(directory / "results.csv"):\n            detector = row.get("ood_method", "lof")\n            prediction_path = directory / ("predictions.csv" if is_deepunk else f"predictions_{detector}.csv")\n            predictions = read(prediction_path)\n            if len(predictions) != 3080:\n                raise ValueError(f"Unexpected test sample count in {prediction_path}")\n            metrics, categories = score_predictions(predictions, "unseen" if is_deepunk else "19")\n            scale = 100 if is_deepunk else 1\n            for metric, value in metrics.items():\n                if abs(float(row[metric]) * scale - value) > .011:\n                    raise ValueError(f"Score differs from predictions in {directory}: {metric}")\n            audit.append(dict(trial=directory.relative_to(output).as_posix(), detector=detector,\n                              test_rows=len(predictions), verified=True))\n            context = dict(trial=directory.relative_to(output).as_posix(), detector=detector)\n            for category in ("correct", "unknown accepted", "known rejected", "wrong known class"):\n                errors.append(dict(context, category=category, count=categories[category]))\n            if is_deepunk and directory.parent.name == "full":\n                folds.append(dict(known_ratio=ratio, fold=fold, **metrics))\n                for index, pred in enumerate(predictions):\n                    if pred["gold"] != pred["predicted"] and index in (0, 2, 41) and ratio == "0.25" and fold == "0":\n                        examples.append(dict(test_row=index, raw_gold=pred["raw_gold"],\n                                             gold=pred["gold"], predicted=pred["predicted"]))\n            if "short_5epoch" in directory.parts:\n                paired.append(dict(variant=directory.parent.name, detector=detector,\n                                   seconds=duration, epoch_cap=epochs, **metrics))\n    if len(folds) != 10 or len(paired) != 4:\n        raise ValueError("Expected ten DeepUNK fold rows and four paired detector rows")\n    for ratio in ("0.25", "0.5"):\n        selected = [row for row in folds if row["known_ratio"] == ratio]\n        for metric in ("K-F1", "N-F1"):\n            values = [row[metric] for row in selected]\n            reference = float(published["deepunk", ratio][metric])\n            comparisons.append(dict(known_ratio=ratio, metric=metric, published=reference,\n                                    mean=round(mean(values), 3), fold_sd=round(stdev(values), 3),\n                                    delta=round(mean(values)-reference, 3)))\n    table_path = output / "tables"\n    for name, rows in [("deepunk_folds", folds), ("deepunk_comparison", comparisons),\n                       ("knncon_paired", paired), ("errors", errors), ("runtime", runtime),\n                       ("prediction_audit", audit), ("failure_examples", examples)]:\n        write(table_path / f"{name}.csv", rows)\n    # Existing mask audit was independently generated against the pinned dataset.\n    mask = read(output / "table_mask_audit.csv")\n    write(table_path / "mask_audit.csv", mask)\n    print(f"Verified {len(audit)} score rows against 3,080 predictions each.")\n    print("Generated all report tables in", table_path)\n\n\ndef manifest(output):\n    """Record SHA-256 for immutable saved evidence and source provenance."""\n    rows=[]\n    for path in sorted(output.rglob("*")):\n        if path.is_file() and "work" not in path.relative_to(output).parts and "tables" not in path.relative_to(output).parts and path.name != "manifest.csv":\n            rows.append(dict(path=path.relative_to(output).as_posix(), sha256=hashlib.sha256(path.read_bytes()).hexdigest()))\n    write(output / "manifest.csv", rows)\n\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--results", type=Path, default=ROOT / "results")\n    args=parser.parse_args()\n    all_tables(args.results)\n    manifest(args.results)\n', 'code/author_runner.py': '"""Run the released BOLT implementations and audit the BANKING77 label mask."""\n\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nfrom statistics import mean, stdev\nimport subprocess\nimport sys\nfrom time import monotonic\nfrom urllib.request import urlretrieve\nimport zipfile\n\nROOT = Path(__file__).resolve().parent\nUPSTREAM = "https://github.com/CNIC-DSL/BOLT.git"\nREVISION = "2c935438b5506febcfd23e499e3de81868236b17"\nRATIOS = ("0.25", "0.5")\nMETHODS = {"deepunk": "DeepUnk/experiment.py", "knncon": "KnnCon/run_main.py"}\nMETRICS = ("K-F1", "N-F1", "ACC", "F1")\n\n\ndef read_rows(path, delimiter=","):\n    """Read the author TSV/CSV files without inferring label-mask types."""\n    with path.open(encoding="utf-8", newline="") as handle:\n        return list(csv.DictReader(handle, delimiter=delimiter))\n\n\ndef write_rows(path, rows):\n    """Write a reproducible table with stable column order."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef prepare(source):\n    """Acquire the immutable author release and the DeepUNK embeddings."""\n    if not source.exists():\n        subprocess.run(["git", "clone", UPSTREAM, str(source)], check=True)\n        subprocess.run(["git", "checkout", "--detach", REVISION], cwd=source, check=True)\n    verify_revision(source)\n    glove = source / "pretrained_models" / "glove.6B.300d.txt"\n    glove.parent.mkdir(exist_ok=True)\n    if not glove.exists():\n        archive = glove.with_suffix(".zip")\n        urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", archive)\n        with zipfile.ZipFile(archive) as embeddings:\n            temporary = glove.with_suffix(".partial")\n            with embeddings.open(glove.name) as original, temporary.open("wb") as target:\n                shutil.copyfileobj(original, target)\n            temporary.replace(glove)\n        archive.unlink()\n\n\ndef verify_revision(source):\n    """Fail if the chosen checkout does not identify the pinned release."""\n    head = (source / ".git" / "HEAD").read_text().strip()\n    if head.startswith("ref: "):\n        ref = head.removeprefix("ref: ")\n        ref_path = source / ".git" / ref\n        if ref_path.exists():\n            head = ref_path.read_text().strip()\n        else:\n            packed = (source / ".git" / "packed-refs").read_text().splitlines()\n            head = next(line.split()[0] for line in packed if line.endswith(" " + ref))\n    if head != REVISION:\n        raise ValueError(f"Expected BOLT {REVISION}, found {head}")\n\n\ndef audit(source):\n    """Verify split alignment and count the effect of the string-to-bool conversion."""\n    data = source / "data" / "banking"\n    rows = []\n    for split in ("train", "dev"):\n        original = read_rows(data / "origin_data" / f"{split}.tsv", "\\t")\n        masked = read_rows(data / "labeled_data/0.1" / f"{split}.tsv", "\\t")\n        if [row["label"] for row in original] != [row["label"] for row in masked]:\n            raise ValueError(f"{split} labels are misaligned")\n        if {row["labeled"] for row in masked} != {"True", "False"}:\n            raise ValueError(f"Unexpected mask values in {split}")\n        for ratio in RATIOS:\n            for fold in range(5):\n                path = data / f"label/fold5/part{fold}/label_known_{ratio}.list"\n                known = set(path.read_text(encoding="utf-8").splitlines())\n                candidates = [row for row in masked if row["label"] in known]\n                selected = [row for row in candidates if row["labeled"] == "True"]\n                if {row["label"] for row in selected} != known:\n                    raise ValueError(f"Missing labeled examples for {ratio}, fold {fold}, {split}")\n                rows.append(dict(split=split, known_ratio=ratio, fold=fold,\n                                 classes=len(known), released_knncon=len(candidates),\n                                 corrected_knncon=len(selected)))\n    return rows\n\n\ndef replace_once(source, needle, replacement):\n    """Anchor instrumentation to the inspected source revision."""\n    if source.count(needle) != 1:\n        raise ValueError(f"Unexpected upstream source around {needle}")\n    return source.replace(needle, replacement)\n\n\ndef stage_method(source, name, corrected, work):\n    """Copy a method to scratch and add prediction exports and the optional mask fix."""\n    entry = Path(METHODS[name])\n    location = work / entry.parent\n    shutil.copytree(source / "code/openset/baselines" / entry.parent, location,\n                    dirs_exist_ok=True)\n    if name == "deepunk":\n        path = location / entry.name\n        code = path.read_text(encoding="utf-8")\n        code = replace_once(code, "    report = classification_report(",\n            \'    os.makedirs(args.save_results_path, exist_ok=True)\\n\'\n            \'    pd.DataFrame({"gold": y_test_mask, "predicted": y_pred, \'\n            \'"raw_gold": df_test["label"]}).to_csv(\\n\'\n            \'        os.path.join(args.save_results_path, "predictions.csv"), index=False\\n\'\n            \'    )\\n\\n    report = classification_report(\')\n    else:\n        path = location / "evaluate.py"\n        code = path.read_text(encoding="utf-8")\n        code = replace_once(code, "                df_to_save = pd.DataFrame([final_results])",\n            \'                pd.DataFrame({"gold": labels_test, "predicted": predict_label_test}).to_csv(\\n\'\n            \'                    os.path.join(self.data_args.save_results_path, \'\n            \'f"predictions_{setting}.csv"), index=False\\n\'\n            \'                )\\n\'\n            \'                df_to_save = pd.DataFrame([final_results])\')\n        code = replace_once(code, \'"seed","ACC","F1","K-F1","N-F1","args"]\',\n                            \'"seed","ACC","F1","K-F1","N-F1","args","ood_method"]\')\n        if corrected:\n            runner = location / entry.name\n            runner_code = runner.read_text(encoding="utf-8")\n            for frame in ("df_train", "df_valid"):\n                runner_code = replace_once(runner_code, f\'{frame}["labeled"].astype(bool)\',\n                                           f\'{frame}["labeled"].eq("True")\')\n            runner.write_text(runner_code, encoding="utf-8")\n    path.write_text(code, encoding="utf-8")\n    return location / entry.name\n\n\ndef run_trial(source, output, name, ratio, fold, mode, epochs=50):\n    """Run one isolated fold and record command, elapsed time, and console output."""\n    corrected = mode == "mask"\n    dest = output / mode / f"{name}_{ratio}_{fold}"\n    dest.mkdir(parents=True, exist_ok=True)\n    work = output / "work" / mode / dest.name\n    args = dict(config=source / f"configs/openset/{name}.yaml", dataset="banking",\n                known_cls_ratio=ratio, labeled_ratio=0.1, fold_idx=fold, fold_num=5,\n                fold_type="fold", seed=2025, num_train_epochs=1 if mode == "quick" else epochs,\n                output_dir=work / "checkpoints", save_results_path=dest)\n    if name == "knncon":\n        args["bert_model"] = "google-bert/bert-base-uncased"\n    entry = stage_method(source, name, corrected, work)\n    interpreter = os.environ.get("BOLT_TF_PYTHON", sys.executable) if name == "deepunk" else sys.executable\n    command = [interpreter, str(entry)]\n    for key, value in args.items():\n        command.extend((f"--{key}", str(value)))\n    env = dict(os.environ, PYTHONHASHSEED="2025", TF_USE_LEGACY_KERAS="1",\n               PYTHONUNBUFFERED="1", WANDB_DISABLED="true")\n    if name == "deepunk" and env.get("BOLT_TF_LIBS"):\n        env["LD_LIBRARY_PATH"] = env["BOLT_TF_LIBS"]\n    if (dest / "run.json").exists():\n        raise FileExistsError(f"Run already completed in {dest}. Choose another output directory.")\n    for path in dest.glob("*.csv"):\n        path.unlink()\n    start = monotonic()\n    print(f"Running {dest.name} ({mode})", flush=True)\n    with (dest / "run.log").open("w", encoding="utf-8") as log:\n        subprocess.run(command, cwd=source, env=env, stdout=log, stderr=subprocess.STDOUT,\n                       check=True)\n    rows = read_rows(dest / "results.csv")\n    expected = 1 if name == "deepunk" else 2\n    if len(rows) != expected:\n        raise ValueError(f"Expected {expected} result rows in {dest}")\n    run_info = dict(command=command, elapsed_seconds=round(monotonic() - start, 2),\n                    upstream=REVISION, mode=mode)\n    (dest / "run.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")\n\n\ndef run_table(output, mode, folds):\n    """Aggregate only complete groups and retain both KNNCon detectors."""\n    rows = []\n    for name in METHODS:\n        if mode == "mask" and name == "deepunk":\n            continue\n        for ratio in RATIOS:\n            by_detector = {}\n            for fold in folds:\n                directory = output / mode / f"{name}_{ratio}_{fold}"\n                if not (directory / "run.json").is_file():\n                    raise ValueError(f"Incomplete trial {directory}")\n                for row in read_rows(directory / "results.csv"):\n                    by_detector.setdefault(row.get("ood_method", "lof"), []).append(row)\n            for detector, trials in by_detector.items():\n                if len(trials) != len(folds):\n                    raise ValueError(f"Missing detector output for {name}, {ratio}, {detector}")\n                result = dict(method=name, known_ratio=ratio, detector=detector,\n                              folds=len(folds))\n                for metric in METRICS:\n                    values = [float(trial[metric]) for trial in trials]\n                    if name == "deepunk":\n                        values = [100 * value for value in values]\n                    result[metric] = round(mean(values), 3)\n                    result[f"{metric}_fold_sd"] = round(stdev(values), 3) if len(values) > 1 else ""\n                rows.append(result)\n    write_rows(output / f"table_{mode}.csv", rows)\n\n\ndef main():\n    """Dispatch preparation, audit, reduced run, full grid, or mask sensitivity."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=["prepare", "audit", "quick", "full", "mask", "check"])\n    parser.add_argument("--upstream", type=Path, default=ROOT / ".bolt")\n    parser.add_argument("--output", type=Path, default=ROOT / "results")\n    options = parser.parse_args()\n    source, output = options.upstream.resolve(), options.output.resolve()\n    if options.action == "prepare":\n        prepare(source)\n        return\n    verify_revision(source)\n    if options.action in {"audit", "check"}:\n        rows = audit(source)\n        if options.action == "audit":\n            write_rows(output / "table_mask_audit.csv", rows)\n        else:\n            import tempfile\n            with tempfile.TemporaryDirectory() as scratch:\n                for method in METHODS:\n                    entry = stage_method(source, method, True, Path(scratch))\n                    compile(entry.read_text(encoding="utf-8"), str(entry), "exec")\n                    if method == "knncon":\n                        evaluator = entry.with_name("evaluate.py")\n                        compile(evaluator.read_text(encoding="utf-8"), str(evaluator), "exec")\n            print(f"Passed split alignment, mask coverage, and patch checks ({len(rows)} split cases)")\n        return\n    if options.action == "quick":\n        run_trial(source, output, "knncon", "0.25", 0, "quick")\n        return\n    folds = range(5)\n    for name in METHODS:\n        if options.action == "mask" and name == "deepunk":\n            continue\n        for ratio in RATIOS:\n            for fold in folds:\n                dest = output / options.action / f"{name}_{ratio}_{fold}"\n                if not (dest / "run.json").exists():\n                    run_trial(source, output, name, ratio, fold, options.action)\n    run_table(output, options.action, folds)\n\n\nif __name__ == "__main__":\n    main()\n', 'code/train.py': '"""Run the completed study protocol using the pinned author implementation."""\nimport argparse\nimport os\nfrom pathlib import Path\nimport subprocess\nfrom urllib.request import urlretrieve\nfrom zipfile import ZipFile\nimport shutil\n\nfrom author_runner import REVISION, UPSTREAM, run_trial, verify_revision\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef acquire(source):\n    """Fetch the recorded author revision without installing model dependencies."""\n    if not source.exists():\n        source.mkdir(parents=True)\n        subprocess.run(["git", "init", str(source)], check=True)\n    head = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=source,\n                          capture_output=True, text=True)\n    if head.returncode != 0:\n        subprocess.run(["git", "fetch", "--depth", "1", UPSTREAM, REVISION],\n                       cwd=source, check=True)\n        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=source,\n                       check=True)\n    verify_revision(source)\n\n\ndef embeddings(source):\n    """Download and extract only the required GloVe embedding file."""\n    target = source / "pretrained_models/glove.6B.300d.txt"\n    target.parent.mkdir(exist_ok=True)\n    if target.exists():\n        return\n    archive = target.parent / "glove.6B.zip"\n    urlretrieve("https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip", archive)\n    partial = target.with_suffix(".partial")\n    with ZipFile(archive) as bundle:\n        with bundle.open(target.name) as original, partial.open("wb") as output:\n            shutil.copyfileobj(original, output)\n    partial.replace(target)\n    archive.unlink()\n\n\ndef main():\n    """Execute one report experiment, retaining previous successful trials."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("experiment", choices=["prepare", "deepunk", "quick", "paired"])\n    parser.add_argument("--upstream", type=Path, default=ROOT / ".bolt")\n    parser.add_argument("--output", type=Path, default=ROOT / "results")\n    args = parser.parse_args()\n    source, output = args.upstream.resolve(), args.output.resolve()\n    acquire(source)\n    os.environ.setdefault("MPLBACKEND", "Agg")\n    if args.experiment == "prepare":\n        return\n    if args.experiment == "deepunk":\n        needed = [(ratio, fold) for ratio in ("0.25", "0.5") for fold in range(5)\n                  if not (output / "full" / f"deepunk_{ratio}_{fold}" / "run.json").exists()]\n        if needed:\n            embeddings(source)\n        for ratio in ("0.25", "0.5"):\n            for fold in range(5):\n                directory = output / "full" / f"deepunk_{ratio}_{fold}"\n                if not (directory / "run.json").exists():\n                    run_trial(source, output, "deepunk", ratio, fold, "full", epochs=50)\n    elif args.experiment == "quick":\n        directory = output / "quick/knncon_0.25_0"\n        if not (directory / "run.json").exists():\n            run_trial(source, output, "knncon", "0.25", 0, "quick", epochs=1)\n    else:\n        for mode in ("full", "mask"):\n            directory = output / "short_5epoch" / mode / "knncon_0.25_0"\n            if not (directory / "run.json").exists():\n                run_trial(source, output / "short_5epoch", "knncon", "0.25", 0, mode, epochs=5)\n\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_reproduction.py': '"""Check the regression in label handling and verify recorded test predictions."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "code"))\nfrom analyze_results import all_tables, score_predictions\n\n\nclass ReproductionChecks(unittest.TestCase):\n    def test_label_mask_regression(self):\n        """Nonempty False strings must never be treated as labeled examples."""\n        mask = ["True", "False", "False", "True"]\n        released = [bool(value) for value in mask]\n        corrected = [value == "True" for value in mask]\n        self.assertEqual(sum(released), 4)\n        self.assertEqual(sum(corrected), 2)\n        from author_runner import stage_method\n        import inspect\n        patch = inspect.getsource(stage_method)\n        self.assertIn(\'.eq("True")\', patch)\n\n    def test_recorded_mask_counts(self):\n        """Check the actual saved BANKING77 mask audit."""\n        from analyze_results import read\n        rows = read(ROOT / "results/table_mask_audit.csv")\n        observed = next(row for row in rows if row["split"] == "train"\n                        and row["known_ratio"] == "0.25" and row["fold"] == "0")\n        self.assertEqual(int(observed["released_knncon"]), 2303)\n        self.assertEqual(int(observed["corrected_knncon"]), 229)\n\n    def test_known_and_unknown_metrics(self):\n        """Check pooled-unknown F1 against a hand-calculated confusion pattern."""\n        rows = [dict(gold="known", predicted="known"), dict(gold="known", predicted="unknown"),\n                dict(gold="unknown", predicted="unknown"), dict(gold="unknown", predicted="known")]\n        metrics, errors = score_predictions(rows, "unknown")\n        self.assertEqual(metrics["K-F1"], 50)\n        self.assertEqual(metrics["N-F1"], 50)\n        self.assertEqual(errors["unknown accepted"], 1)\n        self.assertEqual(errors["known rejected"], 1)\n\n    def test_saved_prediction_evidence(self):\n        """Detect absent outputs, scale errors, or metrics inconsistent with predictions."""\n        all_tables(ROOT / "results")\n\n    def test_analysis_under_work_ancestor(self):\n        """A checkout inside a directory named work must retain its trial evidence."""\n        import shutil\n        import tempfile\n        with tempfile.TemporaryDirectory() as temporary:\n            copied = Path(temporary) / "work" / "checkout" / "results"\n            shutil.copytree(ROOT / "results", copied)\n            all_tables(copied)\n            self.assertTrue((copied / "tables/deepunk_comparison.csv").is_file())\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'requirements-tensorflow.txt': 'absl-py==2.5.1\nastunparse==1.6.3\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpickle==3.1.2\ncontourpy==1.3.3\ncycler==0.12.1\nflatbuffers==25.12.19\nfonttools==4.68.0\ngast==0.7.0\ngoogle-pasta==0.2.0\ngrpcio==1.85.3\nh5py==3.16.0\nidna==3.20\njoblib==1.6.0\nkeras==3.15.1\nkiwisolver==1.5.1\nlibclang==18.1.1\nmarkdown==3.11.3\nmarkdown-it-py==4.3.0\nmarkupsafe==3.0.4\nmatplotlib==3.9.4\nmdurl==0.1.2\nml-dtypes==0.5.4\nnamex==0.1.0\nnltk==3.9.2\nnumpy==1.26.4\nnvidia-cublas-cu12==12.5.3.2\nnvidia-cuda-cupti-cu12==12.5.82\nnvidia-cuda-nvcc-cu12==12.5.82\nnvidia-cuda-nvrtc-cu12==12.5.82\nnvidia-cuda-runtime-cu12==12.5.82\nnvidia-cudnn-cu12==9.3.0.75\nnvidia-cufft-cu12==11.2.3.61\nnvidia-curand-cu12==10.3.6.82\nnvidia-cusolver-cu12==11.6.3.83\nnvidia-cusparse-cu12==12.5.1.3\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.5.82\nopt-einsum==3.4.0\noptree==0.20.0\npackaging==26.3\npandas==2.2.3\npillow==13.0.0\nprotobuf==5.29.6\npygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\npytz==2026.6\npyyaml==6.0.2\nregex==2026.11.18\nrequests==2.34.2\nrich==15.0.0\nscikit-learn==1.5.2\nscipy==1.17.1\nsetuptools==84.2.1\nsix==1.17.0\ntensorboard==2.18.0\ntensorboard-data-server==0.7.2\ntensorflow[and-cuda]==2.18.1\ntensorflow-io-gcs-filesystem==0.37.1\ntermcolor==3.3.0\ntf-keras==2.18.0\nthreadpoolctl==3.8.0\ntqdm==4.70.1\ntyping-extensions==4.16.0\ntzdata==2026.6\nurllib3==2.8.0\nwerkzeug==3.1.9\nwheel==0.48.0\nwrapt==2.5.2\n', 'requirements-torch.txt': 'aiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nattrs==26.1.0\nblinker==1.9.0\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpickle==3.1.2\ncontourpy==1.3.3\ncycler==0.12.1\ndatasets==2.21.0\ndill==0.3.8\ndocopt==0.6.2\nfilelock==4.2.12\nfitlog==0.9.15\nflask==3.1.3\nfonttools==4.68.0\nfrozenlist==1.8.0\nfsspec==2024.6.1\ngitdb==4.0.12\ngitpython==3.2.1\nhf-xet==1.8.0\nhuggingface-hub==0.36.2\nidna==3.20\nitsdangerous==2.2.0\njinja2==3.1.6\njoblib==1.6.0\nkiwisolver==1.5.1\nmarkupsafe==3.0.4\nmatplotlib==3.9.4\nmpmath==1.3.0\nmultidict==7.2.1\nmultiprocess==0.70.16\nnetworkx==3.6.1\nnumpy==1.26.4\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\npackaging==26.3\npandas==2.2.3\npillow==13.0.0\npropcache==0.5.4\npyarrow==17.0.0\npyparsing==3.3.3\npython-dateutil==4.0.0\npytz==2026.6\npyyaml==6.0.2\nregex==2026.11.18\nrequests==2.34.2\nsafetensors==0.8.0\nscikit-learn==1.5.2\nscipy==1.17.1\nsmmap==5.0.3\nsympy==1.13.1\nthreadpoolctl==3.8.0\ntokenizers==0.13.3\ntorch==2.6.0\ntqdm==4.70.1\ntransformers==4.30.2\ntriton==3.2.0\ntyping-extensions==4.16.0\ntzdata==2026.6\nurllib3==2.8.0\nwerkzeug==3.1.9\nxxhash==4.1.0\nyarl==1.25.1\n', 'submission-details.json': '{\n  "group_number": "04",\n  "members": [\n    {\n      "name": "Reem Alabdulla",\n      "id": "26000605",\n      "responsibilities": "Paper selection and literature review",\n      "sections": "1, 2",\n      "code": "Not applicable",\n      "percent": 25,\n      "agreed_name": "Reem Alabdulla"\n    },\n    {\n      "name": "Shaza Mahmod Badr",\n      "id": "26001544",\n      "responsibilities": "Implementation and environment setup",\n      "sections": "3",\n      "code": "train.py; author_runner.py; notebook; requirements",\n      "percent": 25,\n      "agreed_name": "Shaza Mahmod Badr"\n    },\n    {\n      "name": "Amal Alhashmi",\n      "id": "26000656",\n      "responsibilities": "Experiments and validation",\n      "sections": "4",\n      "code": "analyze_results.py; tests/",\n      "percent": 25,\n      "agreed_name": "Amal Alhashmi"\n    },\n    {\n      "name": "Roudha Alblooshi",\n      "id": "26000603",\n      "responsibilities": "Error analysis and report integration",\n      "sections": "5, 6",\n      "code": "analyze_results.py (errors); build_report.py",\n      "percent": 25,\n      "agreed_name": "Roudha Alblooshi"\n    }\n  ],\n  "repository_url": "https://github.com/emailextra/AI504-BOLT-Reproduction",\n  "contributions_confirmed": true\n}'}
for name, source in sources.items():
    target = root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
subprocess.run(["nvidia-smi"], check=True)


## Install and check the GPU environments

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.22"], check=True)

def install(name, packages):
    path = root / name
    python = path / "bin/python"
    if not python.exists():
        subprocess.run(["uv", "venv", "--python", "3.11", str(path)], check=True)
    subprocess.run(
        ["uv", "pip", "install", "--python", str(python), *packages], check=True
    )
    return python

python = install(".venv-torch", (root / "requirements-torch.txt").read_text().splitlines())
tf_python = install(".venv-tf", (root / "requirements-tensorflow.txt").read_text().splitlines())

base_env = dict(os.environ, MPLBACKEND="Agg", TF_USE_LEGACY_KERAS="1",
                CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
if base_env.get("NVIDIA_VISIBLE_DEVICES") in {"none", "void"}:
    base_env.pop("NVIDIA_VISIBLE_DEVICES")
library_dirs = sorted((tf_python.parent.parent / "lib/python3.11/site-packages/nvidia").glob("*/lib"))
assert library_dirs, "TensorFlow CUDA libraries were not installed."
tf_libs = os.pathsep.join([*(str(path) for path in library_dirs),
                           base_env.get("LD_LIBRARY_PATH", "")]).rstrip(os.pathsep)
tf_env = dict(base_env, LD_LIBRARY_PATH=tf_libs)
run_env = dict(base_env, BOLT_TF_PYTHON=str(tf_python), BOLT_TF_LIBS=tf_libs)

torch_check = """
import torch
assert torch.cuda.is_available(), "PyTorch cannot see the GPU"
from datasets import Dataset
print("Dataset import passed:", len(Dataset.from_dict({"text": ["check"]})), flush=True)
a = torch.ones((32, 32), device="cuda")
print("PyTorch GPU:", torch.cuda.get_device_name(), (a @ a).device, flush=True)
"""
tf_check = """
import tensorflow as tf
assert tf.config.list_physical_devices("GPU"), "TensorFlow cannot see the GPU"
tf.config.set_soft_device_placement(False)
with tf.device("/GPU:0"):
    layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, dropout=0.5))
    inputs = tf.ones((4, 16, 8))
    with tf.GradientTape() as tape:
        output = layer(inputs, mask=tf.sequence_mask([16, 12, 8, 4], 16), training=True)
        loss = tf.reduce_sum(output)
    gradients = tape.gradient(loss, layer.trainable_variables)
    assert all(gradient is not None for gradient in gradients)
    assert "GPU:0" in output.device, output.device
    print("TensorFlow LSTM GPU:", output.device, flush=True)
"""
with (results / "environment.txt").open("w") as info:
    for command in (["nvidia-smi"], [str(python), "-VV"],
                    ["uv", "pip", "freeze", "--python", str(python)],
                    ["uv", "pip", "freeze", "--python", str(tf_python)]):
        subprocess.run(command, stdout=info, stderr=subprocess.STDOUT, check=True)

for interpreter, check, env in [(python, torch_check, base_env), (tf_python, tf_check, tf_env)]:
    checked = subprocess.run([str(interpreter), "-c", check], env=env,
                             text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(checked.stdout, flush=True)
    with (results / "environment.txt").open("a") as info:
        info.write(checked.stdout)
    checked.check_returncode()

In [ ]:
def run(script, *args):
    subprocess.run([str(python), str(root / script), *args],
                   cwd=root, env=run_env, check=True)

run("code/train.py", "prepare")
print("Data setup complete.")

## Optional restore of saved evidence
Run this cell only when continuing from an existing results ZIP. Otherwise proceed to the reduced run.

In [ ]:
from io import BytesIO
uploaded = files.upload()
archive = next(iter(uploaded))
allowed = {"full", "quick", "short_5epoch", "tables"}
with ZipFile(BytesIO(uploaded[archive])) as bundle:
    for item in bundle.infolist():
        parts = Path(item.filename).parts
        if not parts or ".." in parts or Path(item.filename).is_absolute():
            continue
        if parts[0] not in allowed and item.filename not in {"table_mask_audit.csv", "environment.txt"}:
            continue
        if item.is_dir() or "work" in parts:
            continue
        target = results.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(bundle.read(item))
print("Completed main trials restored:", len(list((results / "full").glob("*/run.json"))))

## Reduced demonstration
Measured successful runtime was 176.59 seconds on a Tesla T4. Completed restored trials are skipped.

In [ ]:
run("code/train.py", "quick")

## Five partition DeepUNK experiment
The ten successful runs required 19.97 GPU minutes, excluding setup and downloads.

In [ ]:
run("code/train.py", "deepunk")

## Paired KNNCon experiment
Two five-epoch runs at ratio 0.25 on partition zero. Measured combined runtime was 12.03 GPU minutes.

In [ ]:
run("code/train.py", "paired")

## Rebuild tables and run verification

In [ ]:
run("code/author_runner.py", "audit", "--upstream", str(root / ".bolt"),
    "--output", str(results))
run("code/analyze_results.py")
subprocess.run([str(python), "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=root, env=run_env, check=True)
for name in ("deepunk_comparison", "knncon_paired"):
    print((results / "tables" / f"{name}.csv").read_text())

## Save evidence
This cell can run after stopping training. Completed trial outputs are retained and an unfinished trial restarts when resumed.

In [ ]:
with ZipFile("/content/BOLT-final-results.zip", "w") as bundle:
    for path in results.rglob("*"):
        relative = path.relative_to(results)
        if path.is_file() and "work" not in relative.parts:
            bundle.write(path, relative)
    for name in sources:
        bundle.write(root / name, "source/" + name)
files.download("/content/BOLT-final-results.zip")